## 1️⃣ Import Libraries
Import libraries required for differential expression analysis
and result visualization.
- **scanpy** — Wilcoxon rank-sum test via `rank_genes_groups`
- **pandas** — DEG table manipulation and export
- **numpy** — numerical operations
- **matplotlib** — visualization
- **os** — file path management

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

sc.set_figure_params(dpi=100, facecolor="white")

# Output directory for DEG results
DEG_DIR = "../output/DEG_Wilcoxon_T1D_vs_Control"
os.makedirs(DEG_DIR, exist_ok=True)
print(f"✅ Output directory: {DEG_DIR}")

## 2️⃣ Load Annotated Data
Load the fully annotated AnnData object from Script 04 containing
CellTypist cell type labels and raw counts in `adata.layers['counts']`.
Raw counts are required for differential expression analysis.

In [ ]:
adata = sc.read_h5ad("../data/processed/GSE279086_annotated.h5ad")

print(f"Cells     : {adata.shape[0]:,}")
print(f"Genes     : {adata.shape[1]:,}")
print(f"Cell types: {adata.obs['majority_voting'].nunique()}")
print(f"\nDisease distribution:")
print(adata.obs['disease'].value_counts())
print(f"\nCell type distribution:")
print(adata.obs['majority_voting'].value_counts())

## 3️⃣ Differential Expression — T1D vs Lean Control (Per Cell Type)
Run Wilcoxon rank-sum test for each cell type comparing Type 1 Diabetes
cells against Lean Control cells. Analysis is performed per cell type
to identify cell-type-specific transcriptional changes.

### Method: Wilcoxon Rank-Sum Test
- Non-parametric test — no normality assumption required
- Robust to zero-inflation characteristic of scRNA-seq data
- Standard method for scRNA-seq DEG analysis in Python/scanpy
- Equivalent in principle to MAST used in the R pipeline

### Filtering Criteria:
- Minimum 20 cells per group per cell type
- Both T1D and Lean Control must be present
- Results saved per cell type as individual CSV files

In [ ]:
celltypes = adata.obs['majority_voting'].unique()
min_cells = 20
results_summary = []

for ct in sorted(celltypes):
    # Subset to cell type
    adata_ct = adata[adata.obs['majority_voting'] == ct].copy()
    
    # Check both conditions present
    cond_counts = adata_ct.obs['disease'].value_counts()
    
    if 'Type 1 Diabetes' not in cond_counts.index or \
       'Lean Control' not in cond_counts.index:
        print(f"⚠️  {ct} — missing condition → skipping")
        continue
    
    # Check minimum cells
    if cond_counts['Type 1 Diabetes'] < min_cells or \
       cond_counts['Lean Control'] < min_cells:
        print(f"⚠️  {ct} — not enough cells → skipping")
        continue
    
    print(f"\n▶ Processing: {ct}")
    print(f"   T1D cells     : {cond_counts['Type 1 Diabetes']}")
    print(f"   Control cells : {cond_counts['Lean Control']}")
    
    # Set condition as identity
    adata_ct.obs['condition'] = adata_ct.obs['disease']
    
    # Use log-normalized data in X
    sc.tl.rank_genes_groups(
        adata_ct,
        groupby='condition',
        groups=['Type 1 Diabetes'],
        reference='Lean Control',
        method='wilcoxon',
        pts=True
    )
    
    # Extract results
    deg = sc.get.rank_genes_groups_df(
        adata_ct,
        group='Type 1 Diabetes',
        pval_cutoff=None,
        log2fc_min=None
    )
    
    deg['cell_type'] = ct
    
    # Save
    ct_clean = ct.replace('/', '_').replace(' ', '_')
    out_file = os.path.join(DEG_DIR, f"DEG_{ct_clean}_T1D_vs_Control.csv")
    deg.to_csv(out_file, index=False)
    
    # Summary
    sig = deg[(deg['pvals_adj'] < 0.05) & (abs(deg['logfoldchanges']) > 0.25)]
    results_summary.append({
        'Cell Type'     : ct,
        'T1D cells'     : cond_counts['Type 1 Diabetes'],
        'Control cells' : cond_counts['Lean Control'],
        'Total DEGs'    : len(deg),
        'Significant'   : len(sig),
        'Upregulated'   : len(sig[sig['logfoldchanges'] > 0]),
        'Downregulated' : len(sig[sig['logfoldchanges'] < 0])
    })
    
    print(f"   Significant DEGs: {len(sig)} (FDR<0.05, |log2FC|>0.25)")
    print(f"   ✔ Saved: {out_file}")

print("\n🎉 DEG analysis complete!")

## 4️⃣ DEG Summary Table
Compile a summary table of DEG results across all cell types
showing the number of significant, upregulated, and downregulated
genes per cell type.

In [ ]:
summary_df = pd.DataFrame(results_summary)
summary_df = summary_df.sort_values('Significant', ascending=False)
summary_df.index = range(1, len(summary_df) + 1)
summary_df.index.name = "No."

# Save summary
summary_df.to_csv("../output/DEG_summary_table.csv", index=True)
print("✅ Saved: output/DEG_summary_table.csv")
print(f"\nTotal significant DEGs across all cell types: {summary_df['Significant'].sum():,}")
summary_df

## 5️⃣ Volcano Plots
Generate volcano plots for the top cell types showing the relationship
between statistical significance (-log10 adjusted p-value) and
biological effect size (log2 fold change). Each dot represents one gene.

### Volcano Plot Interpretation:
- **X-axis** — log2 fold change (positive = upregulated in T1D)
- **Y-axis** — -log10 adjusted p-value (higher = more significant)
- **Red dots** — significantly upregulated (FDR

In [ ]:
# Top 6 cell types by significant DEGs
top_celltypes = summary_df[summary_df['Significant'] > 0].head(6)['Cell Type'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, ct in enumerate(top_celltypes):
    ct_clean = ct.replace('/', '_').replace(' ', '_')
    deg_file = os.path.join(DEG_DIR, f"DEG_{ct_clean}_T1D_vs_Control.csv")
    deg = pd.read_csv(deg_file)

    # Define significance
    deg['significant'] = 'Not significant'
    deg.loc[(deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'] > 0.25),
            'significant'] = 'Upregulated'
    deg.loc[(deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'] < -0.25),
            'significant'] = 'Downregulated'

    colors = {'Upregulated': '#E74C3C',
              'Downregulated': '#3498DB',
              'Not significant': '#95A5A6'}

    ax = axes[idx]
    for sig, color in colors.items():
        mask = deg['significant'] == sig
        ax.scatter(
            deg.loc[mask, 'logfoldchanges'],
            -np.log10(deg.loc[mask, 'pvals_adj'] + 1e-300),
            c=color, s=3, alpha=0.6, label=sig
        )

    # Add threshold lines
    ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)
    ax.axvline(0.25, color='black', linestyle='--', linewidth=0.8)
    ax.axvline(-0.25, color='black', linestyle='--', linewidth=0.8)

    # Top 5 upregulated gene labels
    top_genes = deg[deg['significant'] == 'Upregulated']\
        .nlargest(5, 'logfoldchanges')
    for _, gene in top_genes.iterrows():
        ax.annotate(
            gene['names'],
            xy=(gene['logfoldchanges'],
                -np.log10(gene['pvals_adj'] + 1e-300)),
            fontsize=6, color='darkred'
        )

    ax.set_title(f"{ct}", fontsize=12, fontweight='bold')
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10 (adj. p-value)")
    ax.legend(fontsize=7, markerscale=3)

plt.suptitle("Volcano Plots — T1D vs Lean Control",
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../figures/volcano_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/volcano_plots.png")

## 6️⃣ Save Final DEG Summary
Save the complete DEG summary table combining results from all
cell types into a single reference file for pathway enrichment
analysis in Script 06.

In [ ]:
# Load and combine all DEG files into one master table
all_degs = []

for ct in sorted(adata.obs['majority_voting'].unique()):
    ct_clean = ct.replace('/', '_').replace(' ', '_')
    deg_file = os.path.join(DEG_DIR, f"DEG_{ct_clean}_T1D_vs_Control.csv")
    
    if os.path.exists(deg_file):
        deg = pd.read_csv(deg_file)
        deg['cell_type'] = ct
        all_degs.append(deg)

# Combine
master_deg = pd.concat(all_degs, ignore_index=True)

# Add significance flag
master_deg['significant'] = (
    (master_deg['pvals_adj'] < 0.05) & 
    (abs(master_deg['logfoldchanges']) > 0.25)
)

# Save master table
master_path = "../output/DEG_master_table.csv"
master_deg.to_csv(master_path, index=False)

size_mb = os.path.getsize(master_path) / (1024 * 1024)

print(f"✅ Saved: {master_path}")
print(f"   Total genes tested : {len(master_deg):,}")
print(f"   Total significant  : {master_deg['significant'].sum():,}")
print(f"   Cell types         : {master_deg['cell_type'].nunique()}")
print(f"   File size          : {size_mb:.1f} MB")